In [1]:
!pip install river pandas kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 41.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [2]:
!kaggle datasets list

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [4]:
!kaggle datasets download -d ealaxi/paysim1
!unzip paysim1.zip

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
Resuming from 98566144 bytes (87819417 bytes left)...
100% 178M/178M [00:00<00:00, 143MB/s]

Archive:  paysim1.zip
  inflating: PS_20174392719_1491204439457_log.csv  


In [5]:
import pandas as pd
import numpy as np
from collections import deque
from river import stream, metrics, preprocessing, forest, drift
import random

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")
df = df.sort_values("step").reset_index(drop=True)

df["type_encoded"]      = df["type"].astype("category").cat.codes
df["orig_balance_diff"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
df["dest_balance_diff"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["amount_ratio"]      = df["amount"] / (df["oldbalanceOrg"] + 1)

X = df.drop(columns=["isFraud","isFlaggedFraud","nameOrig","nameDest","type","step"])
y = df["isFraud"]

print(f"Total rows:  {len(df):,}")
print(f"Fraud rows:  {df['isFraud'].sum():,}")
print(f"Fraud rate:  {df['isFraud'].mean()*100:.3f}%")

# =========================
# TRUE BASELINE — ROSFD paper
# ARF + ADWIN
# =========================
class ROSFD_Baseline:
    name = "ROSFD(paper)"
    def __init__(self):
        self.drift_count = 0
        self.adwin       = drift.ADWIN()
        self.model       = (preprocessing.StandardScaler() |
                           forest.ARFClassifier(n_models=10, seed=42))

    def learn_predict(self, x, y_true):
        y_proba = self.model.predict_proba_one(x).get(1, 0.0)
        y_pred  = self.model.predict_one(x)

        self.adwin.update(int(y_pred != y_true))
        if self.adwin.drift_detected:
            self.drift_count += 1

        self.model.learn_one(x, y_true)

        return y_pred, y_proba

Total rows:  6,362,620
Fraud rows:  8,213
Fraud rate:  0.129%


In [7]:
import pandas as pd
import numpy as np
from collections import deque
from river import stream, metrics, preprocessing, forest, drift
import random

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")
df = df.sort_values("step").reset_index(drop=True)

df["type_encoded"]      = df["type"].astype("category").cat.codes
df["orig_balance_diff"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
df["dest_balance_diff"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["amount_ratio"]      = df["amount"] / (df["oldbalanceOrg"] + 1)

X = df.drop(columns=["isFraud","isFlaggedFraud","nameOrig","nameDest","type","step"])
y = df["isFraud"]

print(f"Total rows:  {len(df):,}")
print(f"Fraud rows:  {df['isFraud'].sum():,}")
print(f"Fraud rate:  {df['isFraud'].mean()*100:.3f}%")

# =========================
# H1 — ARF + ADWIN + Dynamic Resampling Window
# Drift → rebalance window → retrain
# =========================
class H1_DRW:
    name = "H1-DRW"
    def __init__(self, window_size=2000, ratio=200):
        self.window      = deque(maxlen=window_size)
        self.ratio       = ratio
        self.drift_count = 0
        self.adwin       = drift.ADWIN()
        self._build()

    def _build(self):
        self.model = (preprocessing.StandardScaler() |
                     forest.ARFClassifier(n_models=10, seed=42))

    def _rebalance_retrain(self):
        fraud  = [(x,y) for x,y in self.window if y == 1]
        normal = [(x,y) for x,y in self.window if y == 0]
        if not fraud:
            return
        oversampled = (fraud * ((len(normal)//len(fraud)) + 1))[:len(normal)]
        balanced    = normal + oversampled
        random.shuffle(balanced)
        self._build()
        for x, y in balanced:
            self.model.learn_one(x, y)

    def learn_predict(self, x, y_true):
        y_proba = self.model.predict_proba_one(x).get(1, 0.0)
        y_pred  = self.model.predict_one(x)

        self.window.append((x, y_true))
        self.adwin.update(int(y_pred != y_true))

        if self.adwin.drift_detected:
            self.drift_count += 1
            self._rebalance_retrain()
        else:
            self.model.learn_one(x, y_true)
            if y_true == 1:
                for _ in range(self.ratio - 1):
                    self.model.learn_one(x, y_true)

        return y_pred, y_proba

Total rows:  6,362,620
Fraud rows:  8,213
Fraud rate:  0.129%


In [8]:
import pandas as pd
from river import stream, metrics

# STREAM DATASET
dataset = stream.iter_pandas(X, y)

# INIT MODEL
model = H1_DRW()

# metrics
auc = metrics.ROCAUC()
f1 = metrics.F1()
recall = metrics.Recall()
precision = metrics.Precision()

fraud_seen = 0
normal_seen = 0

# RUN STREAM
for i, (x_i, y_i) in enumerate(dataset):

    y_pred, y_proba = model.learn_predict(x_i, y_i)

    # update metrics
    auc.update(y_i, y_proba)
    f1.update(y_i, y_pred)
    recall.update(y_i, y_pred)
    precision.update(y_i, y_pred)

    # count classes
    if y_i == 1:
        fraud_seen += 1
    else:
        normal_seen += 1

    # logging
    if i % 100_000 == 0 and i > 0:
        print(f"\n{'='*50}")
        print(f"Step: {i:,}")
        print(f"Fraud seen: {fraud_seen:,}")
        print(f"Normal seen: {normal_seen:,}")
        print(f"{auc}")
        print(f"{f1}")
        print(f"{recall}")
        print(f"{precision}")
        print(f"Drifts:    {model.drift_count}")

# FINAL RESULT
print(f"\n{'='*50}")
print("FINAL RESULT")
print(f"{'='*50}")
print(f"{auc}")
print(f"{f1}")
print(f"{recall}")
print(f"{precision}")
print(f"Drifts:    {model.drift_count}")
print(f"Fraud seen: {fraud_seen:,}")
print(f"Normal seen: {normal_seen:,}")


Step: 100,000
Fraud seen: 116
Normal seen: 99,885
ROCAUC: 86.26%
F1: 27.72%
Recall: 31.90%
Precision: 24.50%
Drifts:    4

Step: 200,000
Fraud seen: 147
Normal seen: 199,854
ROCAUC: 84.10%
F1: 27.32%
Recall: 34.01%
Precision: 22.83%
Drifts:    5

Step: 300,000
Fraud seen: 181
Normal seen: 299,820
ROCAUC: 82.94%
F1: 27.77%
Recall: 35.36%
Precision: 22.86%
Drifts:    7

Step: 400,000
Fraud seen: 206
Normal seen: 399,795
ROCAUC: 82.09%
F1: 27.77%
Recall: 35.92%
Precision: 22.63%
Drifts:    7

Step: 500,000
Fraud seen: 233
Normal seen: 499,768
ROCAUC: 81.80%
F1: 26.94%
Recall: 36.48%
Precision: 21.36%
Drifts:    11

Step: 600,000
Fraud seen: 361
Normal seen: 599,640
ROCAUC: 85.85%
F1: 36.98%
Recall: 46.81%
Precision: 30.56%
Drifts:    14

Step: 700,000
Fraud seen: 417
Normal seen: 699,584
ROCAUC: 84.73%
F1: 36.02%
Recall: 46.04%
Precision: 29.58%
Drifts:    15

Step: 800,000
Fraud seen: 460
Normal seen: 799,541
ROCAUC: 83.85%
F1: 34.34%
Recall: 45.87%
Precision: 27.44%
Drifts:    17

Step